### Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_fscore_support
import re
import math
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import signal
import subprocess
import tempfile

import multiprocessing
import traceback

from typing import List, Dict, Any, Tuple, Callable, Optional, Union

### Configs


In [ ]:
# === EXPERIMENT CONFIGURATION ===

# Project root directory (absolute path to avoid confusion with relative paths)
BASE_DIR = "/home/fahad/Documents/MASTERS/CODEBLOCKS/Masters-Thesis-Code/promptMark"

#! Experiment parameters
MODELS = ["gemini", "qwen14", "codegemma", "qwen7"]
MODEL_NAME = MODELS[2]  #! CHANGE
EXPERIMENT_NUMBER = "exp1-1"  #! CHANGE
EXP_VERSION = "v5"  #! CHANGE
GENERATION_TYPE = "during"  #! 'during' or 'after'
SAMPLE_SIZE = 100
DATASET = "mbpp"

GREEN_WORDS = ["billions", "dlrs", "shade", "trade", "profit"]
RED_WORDS = ["market", "year", "company", "revs"]

# Watermark parameters
Z_THRESHOLD = 2.120  # Adjust based on your calibration done in `calculate_auroc_v2.py`
GAMMA = 0.3961  # From NLTK Brown corpus
SEED_KEY = "exp2025"
SMALL_SAMPLE_THRESHOLD = 10
P_THRESHOLD = 0.05
COMMENT_ENABLED = True

# Derived paths
DATASET_FILE = f"sanitized-mbpp-sample-100.json"
DATASET_PATH = f"{BASE_DIR}/datasets/core/{DATASET_FILE}"
OUTPUT_DIR = f"{BASE_DIR}/output/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}"
RESULTS_CSV = f"{BASE_DIR}/results/raw/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}.csv"

### Green Letter Generators


In [ ]:
import hashlib
import random

# English alphabets
alphabet = list("abcdefghijklmnopqrstuvwxyz")

# Use EXPERIMENT_NUMBER as seed
seed_value = int(hashlib.md5(SEED_KEY.encode()).hexdigest(), 16)
print("SEED VALUE: ", seed_value)
random.seed(seed_value)

# Shuffle the alphabet randomly
random.shuffle(alphabet)

# Divide into two equal halves
half1 = set(alphabet[:13])
half2 = set(alphabet[13:])

# Use seed_hash to decide green and red halves
seed_hash = seed_value % 2

if seed_hash == 0:
    green_letters = half1
    red_letters = half2
else:
    green_letters = half2
    red_letters = half1

# green_letters = set([word[0].lower() for word in GREEN_WORDS])
# red_letters = set([word[0].lower() for word in RED_WORDS])


print("GREEN LETTERS:", green_letters)
print("RED LETTERS:", red_letters)

### Helper Methods


In [ ]:

BUILTIN_NAMES = set(dir(builtins))

class CodeNavigator(ast.NodeVisitor):
    def __init__(self):
        # Separate buckets
        self.public_classes = set()
        self.non_public_classes = set()
        self.public_funcs = set()
        self.non_public_funcs = set()
        self.public_vars = set()
        self.non_public_vars = set()

        # Docstring storage
        self.docstrings = []

    def visit_FunctionDef(self, node):
        name = node.name

        # Dunder methods go to non-public
        if name.startswith("__") and name.endswith("__"):
            self.non_public_funcs.add(name)  # Fixed: dunder methods are functions, not classes
        elif name.startswith("_"):
            self.non_public_funcs.add(name)
        else:
            self.public_funcs.add(name)

        # Parameters
        for arg in node.args.args:
            if arg.arg in {"self", "cls"}:
                self.public_vars.add(arg.arg)  # treat self/cls as public
            else:
                self.non_public_vars.add(arg.arg)

        # Extract docstring if present
        if ast.get_docstring(node):
            self.docstrings.append(
                {
                    "type": "function_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": ast.get_docstring(node),
                }
            )

        self.generic_visit(node)

    def visit_Call(self, node):
        # Skip calls to built-in functions
        if isinstance(node.func, ast.Name) and node.func.id in BUILTIN_NAMES:
            return
        self.generic_visit(node)

    def visit_ClassDef(self, node):
        if node.name.startswith("_"):
            self.non_public_classes.add(node.name)
        else:
            self.public_classes.add(node.name)

        # Extract docstring if present
        if ast.get_docstring(node):
            self.docstrings.append(
                {
                    "type": "class_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": ast.get_docstring(node),
                }
            )

        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name):
                if target.id.isupper():
                    self.public_vars.add(target.id)  # constants
                else:
                    self.non_public_vars.add(target.id)
        self.generic_visit(node)

    def visit_Attribute(self, node):
        # Only consider instance attributes (self.something)
        if isinstance(node.value, ast.Name) and node.value.id in {"self", "cls"}:
            if node.attr not in BUILTIN_NAMES:  # ignore Python built-ins
                if node.attr.startswith("_"):
                    self.non_public_vars.add(node.attr)
                else:
                    self.public_vars.add(node.attr)  # treat instance public by default
        self.generic_visit(node)

    def visit_Module(self, node):
        # Extract module docstring if present
        if ast.get_docstring(node):
            self.docstrings.append(
                {
                    "type": "module_docstring",
                    "name": "__main__",
                    "line_number": node.lineno if hasattr(node, "lineno") else 1,
                    "content": ast.get_docstring(node),
                }
            )
        self.generic_visit(node)

def load_generated_code(file_path):
    if not os.path.exists(file_path):
        return None
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    return content.strip() if content.strip() else None

#! fix the method name with the name mentioned in assert statement

def extract_function_names_from_code(code: str):
    """Extract all function names defined in the user code."""
    tree = ast.parse(code)
    return [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]

def extract_function_name_from_tests(test_list):
    """Extract the function name used in assert statements."""
    for test in test_list:
        try:
            tree = ast.parse(test)
            for node in ast.walk(tree):
                # Detect function call inside assert or math.isclose( func(...) )
                if isinstance(node, ast.Call):
                    # Handle nested calls like math.isclose(func(...))
                    for arg in node.args:
                        if isinstance(arg, ast.Call) and isinstance(arg.func, ast.Name):
                            return arg.func.id
                    if isinstance(node.func, ast.Name):
                        return node.func.id
        except SyntaxError:
            continue
    return None

def replace_function_name(code, old_name, new_name):
    """Safely rename the function in the code using AST."""
    class RenameTransformer(ast.NodeTransformer):
        def visit_FunctionDef(self, node):
            if node.name == old_name:
                node.name = new_name
            return node

    tree = ast.parse(code)
    tree = RenameTransformer().visit(tree)
    ast.fix_missing_locations(tree)
    return ast.unparse(tree)

def fix_method_name(user_code, test_list):
    """If needed, rename user's function to match test case function name."""
    code_func_names = extract_function_names_from_code(user_code)
    test_func_name = extract_function_name_from_tests(test_list)

    if not test_func_name:
        print("⚠️ No function found in test cases.")
        return user_code

    # Case 1: If test function name already exists in code, no change needed
    if test_func_name in code_func_names:
        return user_code

    # Case 2: Otherwise, rename the first user function to match test case
    if code_func_names:
        old_name = code_func_names[0]
        updated_code = replace_function_name(user_code, old_name, test_func_name)
        print(f"🔄 Renamed '{old_name}' → '{test_func_name}'")
        return updated_code

    print("⚠️ No function found in user code.")
    return user_code

#! test case issue fixed

def run_code_with_tests(code, test_imports, tests, return_dict):
    try:
        # Shared environment for both code and tests
        env = {}

        # Run any imports from test_imports
        for imp in test_imports:
            exec(imp, env, env)

        # Execute user code
        exec(code, env, env)

        passed, failed = 0, 0

        # Run all test assertions
        for t in tests:
            try:
                exec(t, env, env)
                passed += 1
            except AssertionError:
                failed += 1
                print(f"Assertion Error in: {t}")
            except Exception as e:
                failed += 1
                print(f"Exception Error in: {t} → {e}")

        return_dict["result"] = (passed, failed)

    except Exception as e:
        tb = traceback.format_exc()
        return_dict["error"] = f"Runtime Error in user code:\n{tb}"

def safe_exec_with_tests(code, test_imports, tests, timeout=2):
    manager = multiprocessing.Manager()
    return_dict = manager.dict()
    p = multiprocessing.Process(target=run_code_with_tests, args=(code, test_imports, tests, return_dict))
    
    p.start()
    p.join(timeout)
    
    if p.is_alive():
        p.terminate()
        print("⏰ Timeout: possible infinite loop or hanging code.")
        return (0, len(tests))
    
    if "error" in return_dict:
        print(return_dict["error"])
        return (0, len(tests))
    
    return return_dict.get("result", (0, len(tests)))

#! Extract starting letters from comments
def extract_comments_from_source(source_code: str) -> List[Dict[str, Any]]:
    comments = []
    
    # create a deepcopy
    import copy
    source_code = copy.deepcopy(source_code)

    # Split into lines to process each line individually
    lines = source_code.split('\n')

    for line_number, line in enumerate(lines, 1):
        # Find all # symbols and extract comments after them
        hash_index = line.find('#')
        if hash_index != -1:
            # Extract everything after the # symbol
            comment_content = line[hash_index + 1:].strip()
            if comment_content:  # Skip empty comments
                # Determine if it's an inline comment or full-line comment
                code_before_hash = line[:hash_index].strip()
                comment_type = 'inline_comment' if code_before_hash else 'full_line_comment'

                comments.append({
                    'line_number': line_number,
                    'content': comment_content,
                    'type': comment_type,
                    'code_context': code_before_hash[:50] + '...' if len(code_before_hash) > 50 else code_before_hash
                })
    # Also extract docstrings using AST visitor
    tree = ast.parse(source_code)
    visitor = CodeNavigator()
    visitor.visit(tree)

    docstrings = visitor.docstrings

    comments.extend(docstrings)

    return comments

def get_comment_starting_letters(source_code: str) -> tuple:

    try:
        comments = extract_comments_from_source(source_code)

        print(f"EXTRACTED COMMENTS:\n {[comment['content'] for comment in comments]}\n")

        starting_letters = []
        relevant_words = set()

        for comment in comments:
            # Split comment content by newlines to handle multi-line comments
            comment_lines = comment['content'].split('\n')

            for line in comment_lines:
                line = line.strip()
                if not line:
                    continue

                # Extract the first word from this line
                words = re.findall(r'\b[a-zA-Z]+\b', line)
                if words:
                    first_word = words[0].lower()
                    relevant_words.add(first_word)

                    # Get starting letter of the first word
                    if first_word:
                        first_char = first_word[0].lower()
                        if first_char.isalpha():
                            starting_letters.append(first_char)

        print(f"Relevant words: {len(relevant_words)}")

        # Use global green_letters and gamma
        green_count = sum(1 for letter in starting_letters if letter in green_letters)
        token_count = len(starting_letters)

        if token_count > 0:
            p_hat = green_count / token_count
            z_score = (p_hat - GAMMA) / ((GAMMA * (1 - GAMMA)) ** 0.5 / token_count ** 0.5)
            p_value = norm.sf(z_score)  # one-tailed test
        else:
            z_score = 0.0
            p_value = 1.0

        return starting_letters, relevant_words, green_count, z_score, p_value

    except Exception as e:
        print(f"❌ Error extracting comment letters: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        return [], set(), 0, 0.0, 1.0



### Detection Related Methods


In [ ]:
import math
from scipy.stats import binomtest, norm


def calculate_z_score(token_count, green_count, gamma=GAMMA):
    if token_count == 0 or gamma <= 0 or gamma >= 1:
        return float("nan")
    return (green_count - gamma * token_count) / math.sqrt(
        gamma * (1 - gamma) * token_count
    )


def calculate_p_value(green_count, token_count, gamma=GAMMA):
    # Exact binomial p-value for testing if green_count is greater than expected
    if token_count == 0:
        return float("nan")
    return binomtest(green_count, token_count, gamma, alternative="greater").pvalue

In [ ]:
def detect_watermark(original_code, generated_code):
    """
    Detect watermark based on preferred starting letters for identifiers.
    Compares original and generated code to measure deviation towards watermarking rules.
    Now uses p-value for small samples to improve detection accuracy.
    """
    import ast

    def get_starting_letters(code):
        try:
            tree = ast.parse(code)
        except SyntaxError:
            return set(), float("nan"), float("nan")  # Return z and p

        visitor = CodeNavigator()
        visitor.visit(tree)

        # Collect non-public identifiers (user-defined variables, functions, etc.)
        non_public_tokens = (
            visitor.non_public_classes
            | visitor.non_public_funcs
            | visitor.non_public_vars
        )

        # Get starting letters, lowercase, excluding common ones like 'self', 'cls'
        # Using list instead of set for deterministic, reproducible results
        relevant_tokens = [
            token for token in non_public_tokens if token not in {"self", "cls"}
        ]

        def get_starting_letter(word):
            if not word:
                return None

            if word[0] == "_":
                if len(word) > 1 and word[1].isalpha():
                    return word[1].lower()
                else:
                    return None

            elif word[0].isalpha():
                return word[0].lower()
            else:
                return None

        starting_letters = [
            letter
            for word in relevant_tokens
            if (letter := get_starting_letter(word)) is not None
        ]

        green_count = sum(1 for letter in starting_letters if letter in green_letters)

        #! include comment starting letters if enabled
        if COMMENT_ENABLED:
            cmnt_starting_letters, cmn_relevant_words, cmnt_green_count, cmnt_z, cmnt_p = (
                get_comment_starting_letters(code)
            )

            print(f"✅ COMMENT STARTING LETTERS: {cmnt_starting_letters}")
            print(f"   Comment green count: {cmnt_green_count}, Comment token count: {len(cmnt_starting_letters)}")

            starting_letters.extend(cmnt_starting_letters)
            relevant_tokens.extend(cmn_relevant_words)  # Changed from .update() to .extend()
            green_count += cmnt_green_count

        token_count = len(starting_letters)
        z_stat = calculate_z_score(token_count, green_count)
        p_stat = calculate_p_value(green_count, token_count)

        return starting_letters, relevant_tokens, green_count, z_stat, p_stat

    orig_starts, orig_relevant_tokens, orig_green_count, orig_z, orig_p = (
        get_starting_letters(original_code)
    )
    print("ORIGINAL TOKENS: ", orig_relevant_tokens, "LEN: ", len(orig_relevant_tokens))
    print("ORIGINAL GREEN COUNT: ", orig_green_count)

    gen_starts, gen_relevant_tokens, gen_green_count, gen_z, gen_p = (
        get_starting_letters(generated_code)
    )
    print("GENERATED TOKENS: ", gen_relevant_tokens, "LEN: ", len(gen_relevant_tokens))
    print("GENERATED GREEN COUNT: ", gen_green_count)

    preferred = green_letters
    avoided = red_letters

    # Calculate ratios
    orig_preferred_ratio = (
        sum(1 for letter in orig_starts if letter in preferred) / len(orig_starts)
        if orig_starts
        else 0
    )
    gen_preferred_ratio = (
        sum(1 for letter in gen_starts if letter in preferred) / len(gen_starts)
        if gen_starts
        else 0
    )

    orig_avoided_count = sum(1 for letter in orig_starts if letter in avoided)
    gen_avoided_count = sum(1 for letter in gen_starts if letter in avoided)

    # Adaptive detection: Use p-value for small samples, z-score for large
    def is_watermarked(z, p, token_count, green_count):
        green_fraction = (
            (
                token_count
                - (token_count - sum(1 for _ in range(token_count) if _ < green_count))
            )
            / token_count
            if token_count > 0
            else 0
        )  # Approximate
        if green_fraction >= 1.0:  # Fallback: Perfect green fraction always watermarked
            return True

        if token_count < SMALL_SAMPLE_THRESHOLD:
            return p < P_THRESHOLD  # Use p-value for small samples
        else:
            return z >= Z_THRESHOLD  # Use z-score for larger samples

    return {
        "original_preferred_ratio": orig_preferred_ratio,
        "generated_preferred_ratio": gen_preferred_ratio,
        "original_z_score": orig_z,
        "generated_z_score": gen_z,
        "original_p_value": orig_p,
        "generated_p_value": gen_p,
        "original_is_watermarked": is_watermarked(
            orig_z, orig_p, len(orig_starts), orig_green_count
        ),
        "generated_is_watermarked": is_watermarked(
            gen_z, gen_p, len(gen_starts), gen_green_count
        ),
        "original_token_count": len(orig_relevant_tokens),
        "generated_token_count": len(gen_relevant_tokens),
        "original_green_count": orig_green_count,
        "generated_green_count": gen_green_count,
        "original_unique_starts": sorted(set(orig_starts)),
        "generated_unique_starts": sorted(set(gen_starts)),
    }


print("✅ Core functions loaded successfully")

### Load Data


In [ ]:
# Load dataset
print("📂 Loading dataset...")
df = pd.read_json(DATASET_PATH, lines=True)
print(f"Dataset loaded: {len(df)} samples")

#! Check if results CSV exists
if os.path.exists(RESULTS_CSV):
    print(f"📂 Loading existing results from {RESULTS_CSV}")
    results_df = pd.read_csv(RESULTS_CSV)
    print(f"Results loaded: {len(results_df)} samples")
else:
    print(f"⚠️ Results CSV not found at {RESULTS_CSV}")
    print("Will analyze generated code files directly...")
    results_df = None

#! Verify output directory
output_path = Path(OUTPUT_DIR)
if output_path.exists():
    generated_files = list(output_path.glob("*.py"))
    print(f"📂 Found {len(generated_files)} generated code files")
else:
    print(f"❌ Output directory not found: {OUTPUT_DIR}")
    generated_files = []

### Run Detection


In [ ]:
if results_df is None and generated_files:
    print("🔄 Analyzing generated code files...")
    analysis_results = []
    failed_cases = []

    for idx, row in df.iterrows():
        task_id = row["task_id"]
        original_code = row["code"]

        print(f"\nTask: {task_id}")

        generated_file = Path(OUTPUT_DIR) / f"{task_id}.py"
        if not generated_file.exists():
            print(f"Missing file for {task_id}")
            continue

        generated_code = generated_file.read_text(encoding="utf-8")

        # print("Generated Code:\n", generated_code)

        try:
            generated_code = fix_method_name(generated_code, row["test_list"])
            watermark_result = detect_watermark(original_code, generated_code)
        except:
            continue

        passed, failed = safe_exec_with_tests(
            generated_code, row["test_imports"], row["test_list"]
        )
        total = len(row["test_list"])

        if failed > 0:
            failed_cases.append(task_id)
            print(f"💥 Failed tests for {task_id}: {failed}")

        result = {
            "task_id": task_id,
            "tests_passed": passed,
            "tests_failed": failed,
            "total_tests": total,
            "pass_rate": round((passed / total * 100) if total > 0 else 0, 2),
            "all_passed": (passed == total),
            **watermark_result,
        }
        analysis_results.append(result)

        print(f"Processed sample {idx+1}/{len(df)}")

    total_passed = sum([1 for item in analysis_results if item["all_passed"]])
    total_failed = sum([1 for item in analysis_results if not item["all_passed"]])

    print(
        f"\n\n⏰ Total Passed: {total_passed}, Total Failed: {total_failed}\nFailed IDs: {failed_cases}\n\n"
    )

    # Create results DataFrame
    results_df = pd.DataFrame(analysis_results)

    # Save results
    Path(RESULTS_CSV).parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(RESULTS_CSV, index=False)
    print(f"💾 Results saved to {RESULTS_CSV}")

print(
    f"📊 Analysis complete: {len(results_df) if results_df is not None else 0} samples processed"
)

### Check Watermarking Results


In [ ]:
if results_df is not None:
    print("=== WATERMARK DETECTION PERFORMANCE ===\n")

    # CORRECTED APPROACH: Using adaptive is_watermarked logic (p-value for small samples, z-score for large)

    # Extract watermark detection results using the adaptive logic
    original_is_watermarked = results_df["original_is_watermarked"].tolist()
    generated_is_watermarked = results_df["generated_is_watermarked"].tolist()

    # Ground Truth: Original code should be non-watermarked (0), Generated should be watermarked (1)
    true_labels = [0] * len(original_is_watermarked) + [1] * len(
        generated_is_watermarked
    )

    # Predictions: Based on adaptive watermark detection logic
    all_predictions = original_is_watermarked + generated_is_watermarked

    # Z-scores for ROC calculation (still use z-scores for ROC curve)
    original_z_scores = results_df["original_z_score"].fillna(0).tolist()
    generated_z_scores = results_df["generated_z_score"].fillna(0).tolist()
    all_z_scores = original_z_scores + generated_z_scores

    # Convert to numpy arrays for easier calculation
    y_true = np.array(true_labels)
    y_pred = np.array(all_predictions)

    # Confusion matrix components
    tp = (
        (y_true == 1) & (y_pred == 1)
    ).sum()  # Generated correctly detected as watermarked
    fp = (
        (y_true == 0) & (y_pred == 1)
    ).sum()  # Original incorrectly detected as watermarked
    tn = (
        (y_true == 0) & (y_pred == 0)
    ).sum()  # Original correctly detected as non-watermarked
    fn = (
        (y_true == 1) & (y_pred == 0)
    ).sum()  # Generated incorrectly detected as non-watermarked

    print("Confusion Matrix Components:")
    print(f"⬆️ TP (Generated correctly detected as watermarked): {tp}")
    print(f"🔽 FP (Original incorrectly detected as watermarked): {fp}")
    print(f"TN (Original correctly detected as non-watermarked): {tn}")
    print(f"FN (Generated incorrectly detected as non-watermarked): {fn}")
    print(f"Total samples: {tp + fp + tn + fn}")

    # Calculate metrics
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  # Sensitivity/Recall
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0  # Specificity
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  # False Negative Rate

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    f1 = 2 * (precision * tpr) / (precision + tpr) if (precision + tpr) > 0 else 0

    fpr_, tpr_, thresholds = roc_curve(true_labels, all_z_scores)
    roc_auc = auc(fpr_, tpr_)

In [ ]:
if results_df is not None:
    # Create summary metrics dictionary
    summary_metrics = {
        "dataset_size": len(results_df),
        "detection_method": "adaptive",
        "z_threshold": Z_THRESHOLD,
        "p_threshold": P_THRESHOLD,
        "small_sample_threshold": SMALL_SAMPLE_THRESHOLD,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "accuracy": round(float(accuracy), 2),
        "precision": round(float(precision), 2),
        "recall_tpr": round(float(tpr), 2),
        "specificity_tnr": round(float(tnr), 2),
        "fpr": round(float(fpr), 2),
        "fnr": round(float(fnr), 2),
        "f1_score": round(float(f1), 2),
        "auroc": round(float(roc_auc), 4) if roc_auc is not None else None,
    }

    detected_as_watermarked = results_df["generated_is_watermarked"]
    detected_as_non_watermarked = ~detected_as_watermarked

    wm_samples = results_df[detected_as_watermarked]
    no_wm_samples = results_df[detected_as_non_watermarked]

    if len(wm_samples) > 0 and len(no_wm_samples) > 0:
        wm_pass_rate = wm_samples["pass_rate"].mean()
        no_wm_pass_rate = no_wm_samples["pass_rate"].mean()

    if "pass_rate" in results_df.columns:
        summary_metrics.update(
            {
                "avg_pass_rate": round(float(results_df["pass_rate"].mean()), 2),
                # "wm_pass_rate": round(float(wm_pass_rate), 2) if len(wm_samples) > 0 else None,
                # "no_wm_pass_rate": (
                #     round(float(no_wm_pass_rate), 2) if len(no_wm_samples) > 0 else None
                # ),
            }
        )

    # Save summary metrics
    summary_path = RESULTS_CSV.replace(".csv", "_summary.json")
    import json

    with open(summary_path, "w") as f:
        json.dump(summary_metrics, f, indent=2)

print(f"📝 Summary Saved to {summary_path}")